# UAV trajectory + user-scheduling environment

Notebook version of `UAV_traj_env.py` (system model of *UAV.pdf*).

A single-antenna rotary-wing UAV flies at altitude $H_u$ from `start` to `end` over $N$ slots and serves exactly one of $M$ single-antenna ground users per slot (the scheduled user gets the whole bandwidth and power, no interference). Channel: pure distance-dependent path loss $\beta_m[n] = \beta_0\, d_m[n]^{-\alpha}$, no beamforming and no small-scale fading, so the channel is a deterministic function of the UAV position and the (known) user positions.

### MDP (Sec. III-A of the paper)
| | |
|---|---|
| **state** | $s[n] = \{\psi_u[n],\ \beta_m[n]\,(\mathrm{dB}),\ D^{rem}_m[n],\ E^{rem}[n],\ N-n\}$ → $2M+4$ values, each normalised to ~[-1, 1] |
| **action** | $a[n] = \{m^*[n],\ \theta[n],\ v[n]\}$ |
| **reward** | $r[n] = R_{m^*}[n]/R_{norm} - w_{zone}\cdot\#\text{zone entries} - w_{energy}\cdot 1\{E^{rem}<0\}$; at slot $N$ additionally $r_{ter} = -w_{demand}\sum_m D^{rem}_m[N+1]/D_m$ |

### Safety rules applied in `step()`, in this order
1. clip the proposed position to the area $[0, \text{area}]^2$ (11g)
2. cancel a move that ends inside a hazardous zone (+ penalty) (11h)
3. return-to-base override: if the (proposed) position could no longer reach `end` in the remaining moves, fly straight towards `end` at $V_{max}$ instead (11f)

Sections: imports → system configuration → physics core → Gymnasium env → quick checks and plots. The training notebook (`training_quantumPPO.ipynb`) loads this notebook with `%run`, and the quick-check section is skipped when it does.

## 1. Imports

In [ ]:
import numpy as np
import gymnasium as gym
from gymnasium import spaces
from dataclasses import dataclass

## 2. System configuration

Network, mission, demand, hazardous zones, rotary-wing propulsion (Zeng, Xu, Zhang 2019) and reward weights in one dataclass.

In [ ]:
@dataclass
class SystemConfig:
    # --- Network ---
    M: int = 3                        # Number of users (M qubits for scheduling)
    user_positions: tuple = ((20.0, 40.0), (85.0, 75.0), (60,20))
                                      # Single-antenna UAV, no beamforming, no fading
    area: float = 100.0               # 100 m x 100 m area
    H_u: float = 25.0                 # UAV altitude
    beta0: float = 6.79e-7            # Reference channel gain at 1 m (-61.7 dB)
    alpha: float = 2.5                # Path-loss exponent (> 2: debris / collapsed buildings)
    B_max: float = 1e6                # Bandwidth 1 MHz (each scheduled user uses all of it)
    noise_psd: float = 1e-18          # Noise PSD -> sigma2 = noise_psd * B_max = -90 dBm
    P_max: float = 10 ** (35 / 10) / 1e3  # UAV transmit power: 35 dBm = 3.16 W

    # --- Mission / trajectory ---
    N: int = 50                       # Number of time slots
    delta: float = 1.0                # Slot length (s) -> mission T = N * delta = 60 s
    V_max: float = 10.0               # Max UAV speed (m/s)
    start: tuple = (0.0, 0.0)         # Initial position psi_I
    end: tuple = (100.0, 100.0)       # Final position psi_F

    # --- Demand ---
    D_min: float = 90e6              # Per-user demand ~ U[D_min, D_max] bits
    D_max: float = 130e6              # mean total 400 Mb vs ~566 Mb if always overhead

    # --- Hazardous zones (users sampled outside them) ---
    J: int = 3
    zone_centers: tuple = ((60.0, 60.0), (90.0, 10.0), (90, 50))   # scaled 1/5 with the area
    zone_radius: tuple = (10.0, 6.0, 8.0)

    # --- Rotary-wing propulsion (Zeng, Xu, Zhang 2019) ---
    P0: float = 79.86                 # Blade profile power (W)
    Pi: float = 88.63                 # Induced power in hover (W)
    U_tip: float = 120.0              # Rotor tip speed (m/s)
    v0: float = 4.03                  # Mean rotor induced velocity (m/s)
    d0: float = 0.6                   # Fuselage drag ratio
    rho: float = 1.225                # Air density (kg/m^3)
    s: float = 0.05                   # Rotor solidity
    A: float = 0.503                  # Rotor disc area (m^2)
    E_max: float = 7.5e3              # Battery budget (J)

    # --- Reward ---
    R_norm: float = 1e7               # Rate normalisation (10 Mbit/s -> rate term <= ~0.94 per slot)
    w_zone: float = 1.0               # Penalty per hazardous-zone entry (move is also cancelled)
    w_energy: float = 5.0             # Penalty per slot while the battery is depleted
    w_demand: float = 30.0            # Terminal penalty weight for unmet demand

    # --- Episode options ---
    resample_users: bool = False      # False: user positions fixed for all episodes
    resample_demands: bool = False    # False: same demands D_m every episode

## 3. Physics core

Geometry, channel gain (Eq. 2), achievable rate (Eqs. 5–6) and propulsion power (Eq. 9).

In [ ]:
class UAVTrajectoryCore:
    """Physics of the system model: geometry, channel, rate, propulsion power."""

    def __init__(self, cfg: SystemConfig, seed: int = 0):
        self.cfg = cfg
        self.rng = np.random.default_rng(seed)
        self.zone_c = np.asarray(cfg.zone_centers, dtype=np.float64).reshape(-1, 2)[: cfg.J]
        self.zone_r = np.asarray(cfg.zone_radius, dtype=np.float64)[: cfg.J]
        self.sigma2 = cfg.noise_psd * cfg.B_max

    # ---------- users / demands ----------
    def in_zone(self, xy):
        """Boolean array (J,) : is point xy strictly inside hazardous zone j."""
        d = np.linalg.norm(self.zone_c - np.asarray(xy)[None, :], axis=1)
        return d < self.zone_r

    def generate_users(self):
        """Uniform user positions in the area, rejected if inside a hazardous zone."""

        if self.cfg.user_positions is not None:
            return np.asarray(self.cfg.user_positions, dtype=np.float64).reshape(self.cfg.M, 2)
        cfg = self.cfg
        users = np.zeros((cfg.M, 2))
        for m in range(cfg.M):
            while True:
                p = self.rng.uniform(0.0, cfg.area, size=2)
                if not np.any(self.in_zone(p)):
                    users[m] = p
                    break
        return users

    def generate_demands(self):
        cfg = self.cfg
        return self.rng.uniform(cfg.D_min, cfg.D_max, size=cfg.M)

    # ---------- channel ----------
    def distances(self, uav_xy, user_xy):
        """Eq. (1): 3D UAV-user distances."""
        dx = user_xy[:, 0] - uav_xy[0]
        dy = user_xy[:, 1] - uav_xy[1]
        return np.sqrt(dx ** 2 + dy ** 2 + self.cfg.H_u ** 2)

    def channel_gain(self, uav_xy, user_xy):
        """Eq. (2): beta_m[n] = beta0 * d_m[n]^-alpha (path loss only, no fading).

        Returns the channel power gain of every user, plus the distances.
        """
        cfg = self.cfg
        d = self.distances(uav_xy, user_xy)
        beta = cfg.beta0 * d ** (-cfg.alpha)
        return beta, d

    def compute_rate(self, beta):
        """Eqs. (5)-(6): rate of every user *if* it were scheduled (bits/s)."""
        cfg = self.cfg
        snr = cfg.P_max * beta / self.sigma2
        return cfg.B_max * np.log2(1.0 + snr), snr

    # ---------- energy ----------
    def propulsion_power(self, V):
        """Eq. (9): rotary-wing propulsion power P(V) in W."""
        c = self.cfg
        blade = c.P0 * (1.0 + 3.0 * V ** 2 / c.U_tip ** 2)
        induced = c.Pi * np.sqrt(np.sqrt(1.0 + V ** 4 / (4.0 * c.v0 ** 4)) - V ** 2 / (2.0 * c.v0 ** 2))
        parasite = 0.5 * c.d0 * c.rho * c.s * c.A * V ** 3
        return blade + induced + parasite

## 4. Gymnasium environment

One episode = $N$ slots. The action is either a tuple `(m_star, [theta_raw, v_raw])` or a flat array `[m_star, theta_raw, v_raw]`:

- $\theta = \pi\cdot\theta_{raw}$ (wrapped to $[0, 2\pi)$)
- $v = (\mathrm{clip}(v_{raw}, -1, 1) + 1)/2 \in [0, 1]$

In [ ]:
class UAVTrajectoryEnv(gym.Env):
    """Gymnasium env. One episode = N slots.

    Action: either a tuple (m_star, cont) with cont = [theta_raw, v_raw], or a
    flat array [m_star, theta_raw, v_raw].
        theta = pi * theta_raw  (wrapped to [0, 2pi); any real number is valid)
        v     = (clip(v_raw, -1, 1) + 1) / 2  in [0, 1]
    """

    metadata = {"render_modes": ["human"]}

    def __init__(self, cfg: SystemConfig = None, seed: int = None,
                 resample_users: bool = None, resample_demands: bool = None):
        super().__init__()
        self.cfg = cfg if cfg is not None else SystemConfig()
        if resample_users is not None:
            self.cfg.resample_users = resample_users
        if resample_demands is not None:
            self.cfg.resample_demands = resample_demands
        self.core = UAVTrajectoryCore(self.cfg, seed=0 if seed is None else seed)

        M = self.cfg.M
        self.state_dim = 2 * M + 4
        self.n_users = M              # size of the categorical head
        self.cont_dim = 2             # (theta, v)
        self.action_space = spaces.Tuple((
            spaces.Discrete(M),
            spaces.Box(low=-1.0, high=1.0, shape=(2,), dtype=np.float32),
        ))
        self.observation_space = spaces.Box(low=-1.5, high=1.5, shape=(self.state_dim,), dtype=np.float32)

        self.start = np.asarray(self.cfg.start, dtype=np.float64)
        self.end = np.asarray(self.cfg.end, dtype=np.float64)
        self.step_len = self.cfg.V_max * self.cfg.delta     # max distance per slot

        # dB range used to normalise beta_m: from "user right below the UAV"
        # (d = H_u) to "opposite corner" (d = diagonal); mapped onto [-1, 1].
        c = self.cfg
        d_far = np.sqrt(2.0 * c.area ** 2 + c.H_u ** 2)
        self._bdb_hi = 10 * np.log10(c.beta0 * c.H_u ** (-c.alpha))
        self._bdb_lo = 10 * np.log10(c.beta0 * d_far ** (-c.alpha))

        # Users / demands are drawn once here; reset() redraws them if requested.
        self.user_xy = self.core.generate_users()
        self.D = self.core.generate_demands()

        # Episode state
        self.n = None            # current slot index, 1..N
        self.uav_xy = None
        self.D_rem = None
        self.E_rem = None
        self.beta = None
        self.d = None
        self.trajectory = None
        self.state = None

    # ------------------------------------------------------------------ #
    def _normalize_state(self):
        c = self.cfg
        pos = 2.0 * self.uav_xy / c.area - 1.0
        b_db = 10.0 * np.log10(self.beta + 1e-30)
        b_norm = 2.0 * (b_db - self._bdb_lo) / (self._bdb_hi - self._bdb_lo) - 1.0
        b_norm = np.clip(b_norm, -1.0, 1.0)
        dem = 2.0 * self.D_rem / c.D_max - 1.0
        energy = np.clip(2.0 * self.E_rem / c.E_max - 1.0, -1.5, 1.0)
        slots = 2.0 * (c.N - self.n) / c.N - 1.0
        return np.concatenate([pos, b_norm, dem, [energy, slots]]).astype(np.float32)

    def _decode_action(self, action):
        if isinstance(action, (tuple, list)) and len(action) == 2:
            m_star = int(np.asarray(action[0]).reshape(-1)[0])
            cont = np.asarray(action[1], dtype=np.float64).reshape(-1)
        else:
            a = np.asarray(action, dtype=np.float64).reshape(-1)
            m_star, cont = int(round(a[0])), a[1:3]
        m_star = int(np.clip(m_star, 0, self.cfg.M - 1))
        theta = np.mod(np.pi * cont[0], 2.0 * np.pi)
        v = 0.5 * (np.clip(cont[1], -1.0, 1.0) + 1.0)
        return m_star, theta, v

    def _remaining_reach(self):
        """Distance the UAV can still cover after this slot's move."""
        return (self.cfg.N - self.n - 1) * self.step_len

    def _move(self, theta, v):
        """Apply Eq. (16) plus the three safety rules. Returns new pos and flags."""
        c = self.cfg
        old = self.uav_xy.copy()
        zone_hits = 0
        forced = False

        if self.n >= c.N:                       # no move in the last slot
            return old, 0, False

        # proposed move, Eq. (16)
        new = old + v * self.step_len * np.array([np.cos(theta), np.sin(theta)])
        # rule 1: clip to the area (11g)
        new = np.clip(new, 0.0, c.area)
        # rule 2: cancel moves into hazardous zones (11h)
        hit = self.core.in_zone(new)
        if np.any(hit):
            zone_hits += int(np.sum(hit))
            new = old.copy()
        # rule 3: return-to-base override (11f). Paper's trigger
        # ||psi_u[n] - psi_F|| >= (N-n-1) V_max delta, applied to the proposed
        # position so that arrival at psi_F at slot N is guaranteed.
        reach = self._remaining_reach()
        if np.linalg.norm(new - self.end) > reach + 1e-6:
            forced = True
            to_end = self.end - old
            dist = np.linalg.norm(to_end)
            step = min(self.step_len, dist)
            new = old + (to_end / dist) * step if dist > 1e-9 else old.copy()
            # Arrival has priority over zone avoidance; still charge the penalty.
            hit = self.core.in_zone(new)
            zone_hits += int(np.sum(hit))
        return new, zone_hits, forced

    def _reward(self, rate_served, zone_hits, terminal):
        c = self.cfg
        rate_term = rate_served / c.R_norm
        zone_term = -c.w_zone * zone_hits
        energy_term = -c.w_energy * float(self.E_rem < 0.0)
        ter_term = -c.w_demand * float(np.sum(self.D_rem / self.D)) if terminal else 0.0
        reward = rate_term + zone_term + energy_term + ter_term
        return float(reward), float(rate_term), float(zone_term), float(energy_term), float(ter_term)

    # ------------------------------------------------------------------ #
    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        if seed is not None:
            self.core.rng = np.random.default_rng(seed)
        options = options or {}

        if options.get("resample_users", self.cfg.resample_users):
            self.user_xy = self.core.generate_users()
        if options.get("resample_demands", self.cfg.resample_demands):
            self.D = self.core.generate_demands()

        self.n = 1
        self.uav_xy = self.start.copy()
        self.D_rem = self.D.copy()
        self.E_rem = float(self.cfg.E_max)
        self.trajectory = [self.uav_xy.copy()]
        self.beta, self.d = self.core.channel_gain(self.uav_xy, self.user_xy)
        self.state = self._normalize_state()
        return self.state.copy(), {"user_xy": self.user_xy.copy(), "D": self.D.copy()}

    def step(self, action):
        c = self.cfg
        m_star, theta, v = self._decode_action(action)

        # --- 4a. transmission in slot n (channel gain at the current position) ---
        rates, snr = self.core.compute_rate(self.beta)
        rate_served = float(rates[m_star])                # only a_{m*}[n] = 1
        delivered = min(c.delta * rate_served, self.D_rem[m_star])
        self.D_rem[m_star] = max(0.0, self.D_rem[m_star] - c.delta * rate_served)   # Eq. (13)

        # --- 3. movement with safety rules ---
        old_xy = self.uav_xy.copy()
        new_xy, zone_hits, forced = self._move(theta, v)
        V = np.linalg.norm(new_xy - old_xy) / c.delta
        # --- 4b. energy, Eq. (14) (propulsion counted for slots 1..N-1, Eq. (10)) ---
        P_fly = float(self.core.propulsion_power(V)) if self.n < c.N else 0.0
        self.E_rem -= c.delta * P_fly
        self.uav_xy = new_xy

        # --- 5. reward / termination ---
        terminal = self.n >= c.N
        reward, rate_term, zone_term, energy_term, ter_term = self._reward(rate_served, zone_hits, terminal)

        info = {
            "slot": self.n,
            "served_user": m_star,
            "theta": theta,
            "v": v,
            "speed": V,
            "forced_return": forced,
            "zone_hits": zone_hits,
            "rate_Mbps": rate_served / 1e6,
            "all_rates_Mbps": rates / 1e6,
            "snr_dB": 10 * np.log10(snr + 1e-30),
            "delivered_Mb": delivered / 1e6,
            "D_rem_Mb": self.D_rem / 1e6,
            "P_fly_W": P_fly,
            "E_rem_J": self.E_rem,
            "battery_out": self.E_rem < 0.0,
            "uav_xy": self.uav_xy.copy(),
            "dist_to_end": float(np.linalg.norm(self.uav_xy - self.end)),
            "reward": reward,
            "rate_term": rate_term,
            "zone_term": zone_term,
            "energy_term": energy_term,
            "terminal_term": ter_term,
        }
        if terminal:
            info["demand_satisfied"] = self.D_rem <= 1e-6
            info["unmet_fraction"] = float(np.sum(self.D_rem / self.D))

        self.trajectory.append(self.uav_xy.copy())
        if not terminal:
            self.n += 1
            # channel gains at the new position (slot n+1)
            self.beta, self.d = self.core.channel_gain(self.uav_xy, self.user_xy)
        self.state = self._normalize_state()

        return self.state.copy(), reward, terminal, False, info

## 5. Quick checks

Sanity checks and plots of the default scenario. Skipped when this notebook is loaded from the training notebook (`_SKIP_ENV_DEMO = True`).

In [ ]:
RUN_DEMO = not globals().get("_SKIP_ENV_DEMO", False)
if RUN_DEMO:
    import matplotlib.pyplot as plt
    get_ipython().run_line_magic("matplotlib", "inline")

### 5.1 Scenario: users, demands, observation size

In [ ]:
if RUN_DEMO:
    env = UAVTrajectoryEnv(SystemConfig(), seed=42)
    obs, info = env.reset()
    print("state_dim :", env.state_dim, "| users M:", env.cfg.M, "| slots N:", env.cfg.N)
    for m, (xy, D) in enumerate(zip(info["user_xy"], info["D"])):
        print(f"  U{m}: position {tuple(np.round(xy, 1).tolist())}, demand {D / 1e6:.1f} Mb")
    print("initial obs:", np.round(obs, 3))

### 5.2 Propulsion power vs. speed

Hovering costs more than cruising at moderate speed, which is why the energy budget matters for the trajectory.

In [ ]:
if RUN_DEMO:
    V = np.linspace(0, 30, 301)
    P = env.core.propulsion_power(V)
    print(f"P(0) hover = {P[0]:.1f} W | P(V_max = {env.cfg.V_max:.0f} m/s) = "
          f"{env.core.propulsion_power(env.cfg.V_max):.1f} W | min {P.min():.1f} W at {V[P.argmin()]:.1f} m/s")
    plt.figure(figsize=(6, 3.5))
    plt.plot(V, P)
    plt.axvline(env.cfg.V_max, ls="--", c="gray", label="V_max")
    plt.xlabel("speed V (m/s)"); plt.ylabel("propulsion power P(V) (W)")
    plt.title("Rotary-wing propulsion power, Eq. (9)"); plt.grid(True); plt.legend()
    plt.show()

### 5.3 One random-policy episode

Random user choice and random heading/speed, to check the safety rules (area clipping, zone cancelling, return-to-base) and the reward terms.

In [ ]:
if RUN_DEMO:
    rng = np.random.default_rng(0)
    obs, _ = env.reset()
    total_r, served, done = 0.0, [], False
    while not done:
        action = (int(rng.integers(env.cfg.M)), rng.uniform(-1, 1, size=2))
        obs, r, term, trunc, info = env.step(action)
        total_r += r; served.append(info["served_user"]); done = term or trunc
    traj = np.asarray(env.trajectory)
    print(f"return {total_r:.2f} | users fully served {int(info['demand_satisfied'].sum())}/{env.cfg.M} | "
          f"energy left {info['E_rem_J']:.0f} J | final dist to end {info['dist_to_end']:.2f} m")
    print("remaining demand (Mb):", np.round(info["D_rem_Mb"], 1))

In [ ]:
if RUN_DEMO:
    c = env.cfg
    fig, ax = plt.subplots(figsize=(6, 6))
    for (cx, cy), r in zip(env.core.zone_c, env.core.zone_r):
        ax.add_patch(plt.Circle((cx, cy), r, color="red", alpha=0.25))
    colors = plt.cm.tab10(np.arange(c.M) % 10)
    for m, (ux, uy) in enumerate(env.user_xy):
        ax.scatter(ux, uy, marker="^", s=120, color=colors[m], edgecolor="k", zorder=3)
        ax.annotate(f"U{m}", (ux + 3, uy + 3))
    ax.plot(traj[:, 0], traj[:, 1], "-", color="gray", lw=1)
    ax.scatter(traj[:-1, 0], traj[:-1, 1], c=[colors[m] for m in served], s=18, zorder=4)
    ax.scatter(*c.start, marker="s", s=100, color="green", label="start")
    ax.scatter(*c.end, marker="*", s=180, color="gold", edgecolor="k", label="end")
    ax.set_xlim(-10, c.area + 10); ax.set_ylim(-10, c.area + 10); ax.set_aspect("equal")
    ax.set_title("Random policy trajectory (dot colour = served user, red = hazardous zones)")
    ax.legend(loc="upper left"); ax.grid(True)
    plt.show()